# Model Training & Comparison

Set up data loaders, instantiate each of the four decoder architectures
(GRU, CNN-LSTM, Transformer, CNN-Transformer), run a short training loop,
plot training curves, and compare sample predictions.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import load_config, get_default_config
from src.data.loader import load_willett_dataset
from src.data.dataset import NeuralTrialDataset, create_dataloaders
from src.models.gru_decoder import GRUDecoder
from src.models.cnn_lstm import CNNLSTM
from src.models.transformer import TransformerDecoder
from src.models.cnn_transformer import CNNTransformer
from src.training.trainer import Trainer
from src.decoding.greedy import greedy_decode

## 2. Load Configuration and Data

In [ ]:
cfg = get_default_config()
trial_index = load_willett_dataset(cfg)
print(f"Loaded {len(trial_index)} trials")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 3. Set Up Data Loaders

Create training and validation dataloaders from the trial index.

In [ ]:
train_loader, val_loader = create_dataloaders(trial_index, cfg)
print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

# Inspect one batch
for batch in train_loader:
    print(f"\nSample batch:")
    print(f"  signals shape:  {batch['signals'].shape}")
    print(f"  targets shape:  {batch['targets'].shape}")
    print(f"  sig_lengths:    {batch['signal_lengths'][:5]}")
    print(f"  target_lengths: {batch['target_lengths'][:5]}")
    break

## 4. Define GRU Decoder Model

Instantiate the baseline GRU-based decoder.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gru_model = GRUDecoder(
    input_size=cfg.n_channels,
    hidden_size=cfg.hidden_size,
    num_layers=cfg.num_layers,
    num_classes=cfg.num_classes,
).to(device)

total_params = sum(p.numel() for p in gru_model.parameters())
print(f"GRU Decoder: {total_params:,} parameters")
print(gru_model)

## 5. Training Loop Example

Run a short training loop (a few epochs) to demonstrate the pipeline.
In practice, use the full Trainer class for production training.

In [ ]:
trainer = Trainer(
    model=gru_model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=cfg,
    device=device,
)

# Train for a few epochs as a demonstration
num_demo_epochs = 3
history = trainer.train(num_epochs=num_demo_epochs)
print(f"\nTraining complete ({num_demo_epochs} epochs)")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val   loss: {history['val_loss'][-1]:.4f}")

## 6. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs, history['val_loss'], 's-', label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CTC Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if 'val_cer' in history:
    axes[1].plot(epochs, history['val_cer'], 'o-', color='C2', label='Val CER')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('CER')
    axes[1].set_title('Character Error Rate')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'CER not tracked\nin demo mode',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

## 7. Compare Model Architectures

Instantiate all four models and compare parameter counts and forward-pass latency.

In [ ]:
import time

model_classes = {
    'GRU': GRUDecoder,
    'CNN-LSTM': CNNLSTM,
    'Transformer': TransformerDecoder,
    'CNN-Transformer': CNNTransformer,
}

results = []
dummy_input = torch.randn(1, 100, cfg.n_channels).to(device)

for name, ModelClass in model_classes.items():
    model = ModelClass(
        input_size=cfg.n_channels,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        num_classes=cfg.num_classes,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    # Measure forward pass latency
    model.eval()
    with torch.no_grad():
        start = time.perf_counter()
        for _ in range(10):
            _ = model(dummy_input)
        elapsed = (time.perf_counter() - start) / 10

    results.append({'Model': name, 'Params': n_params, 'Latency (ms)': elapsed * 1000})
    print(f"{name:20s}  |  {n_params:>10,} params  |  {elapsed*1000:.2f} ms")

# Bar chart of parameter counts
fig, ax = plt.subplots(figsize=(8, 4))
names = [r['Model'] for r in results]
params = [r['Params'] for r in results]
ax.barh(names, params, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
ax.set_xlabel('Parameter Count')
ax.set_title('Model Architecture Comparison')
plt.tight_layout()
plt.show()

## 8. Sample Predictions

Run the trained GRU model on a few validation samples and display decoded text.

In [ ]:
gru_model.eval()

for batch in val_loader:
    signals = batch['signals'].to(device)
    targets = batch['targets']
    sig_lengths = batch['signal_lengths']

    with torch.no_grad():
        log_probs = gru_model(signals)

    # Decode first few samples in the batch
    for i in range(min(5, signals.shape[0])):
        decoded = greedy_decode(log_probs[i])
        target_text = targets[i] if isinstance(targets[i], str) else f"target_{i}"
        print(f"  Target:  '{target_text}'")
        print(f"  Decoded: '{decoded}'")
        print()
    break

---

**Next:** Proceed to `05_evaluation.ipynb` for evaluation and ablation studies.